# Visualización de predicciones

Carga un volumen CT y la máscara predicha por UNet 2D o 3D y permite explorarlos.

**Columnas mostradas:**
- CT en escala de grises
- Máscara predicha
- Overlay (CT + predicción en rojo)
- Máscara real (opcional, si se proporciona)
- Overlay real (opcional)

**Requisitos:** `numpy`, `matplotlib`, `ipywidgets`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact
import os

## Configuración — edita estas rutas

In [ ]:
# Ruta al volumen CT preprocesado (H, W, D) float32, valores en [0, 1]
# Si tienes el CT en HU (raw), usa apply_windowing=True más abajo
CT_PATH = "output/preprocessed/test/CT/LIDC-IDRI-0078.npy"

# Ruta a la máscara predicha (H, W, D) uint8, valores 0/1
PRED_MASK_PATH = "output/predictions/LIDC-IDRI-0078_mask.npy"

# (Opcional) Ruta a la máscara real ground truth (H, W, D) uint8
# Pon None si no tienes GT
GT_MASK_PATH = "output/preprocessed/test/masks/LIDC-IDRI-0078.npy"

# (Opcional) Ruta al volumen de probabilidades (H, W, D) float32
# Pon None si no guardaste probabilidades al predecir
PROBS_PATH = None

# Si el CT es raw en unidades HU (no preprocesado), activar windowing
APPLY_WINDOWING = False
HU_MIN, HU_MAX = -1000, 600

## Carga de datos

In [ ]:
def apply_windowing(ct, hu_min, hu_max):
    ct = np.clip(ct, hu_min, hu_max)
    return ((ct - hu_min) / (hu_max - hu_min)).astype(np.float32)

ct = np.load(CT_PATH).astype(np.float32)
if APPLY_WINDOWING:
    ct = apply_windowing(ct, HU_MIN, HU_MAX)

pred = np.load(PRED_MASK_PATH)
gt   = np.load(GT_MASK_PATH) if GT_MASK_PATH and os.path.exists(GT_MASK_PATH) else None
prob = np.load(PROBS_PATH)    if PROBS_PATH   and os.path.exists(PROBS_PATH)   else None

assert ct.ndim == 3 and pred.ndim == 3, "CT y máscara deben ser volúmenes 3D (H, W, D)"
assert ct.shape == pred.shape, f"Forma CT {ct.shape} ≠ forma predicción {pred.shape}"

D = ct.shape[2]  # número de cortes en eje Z

print(f"CT shape:   {ct.shape}  | min={ct.min():.3f}  max={ct.max():.3f}")
print(f"Pred shape: {pred.shape} | vóxeles positivos: {pred.sum()}")
if gt is not None:
    inter = np.logical_and(pred > 0.5, gt > 0.5).sum()
    denom = (pred > 0.5).sum() + (gt > 0.5).sum()
    dice  = (2 * inter / denom) if denom > 0 else float('nan')
    print(f"GT shape:   {gt.shape}  | vóxeles positivos: {gt.sum()}")
    print(f"Dice (pred vs GT): {dice:.4f}")
if prob is not None:
    print(f"Probs shape: {prob.shape} | min={prob.min():.3f} max={prob.max():.3f}")

## Navegador interactivo de cortes (eje Z)

In [ ]:
def show_slice(z=0):
    has_gt   = gt   is not None
    has_prob = prob is not None

    n_cols = 3 + (2 if has_gt else 0) + (1 if has_prob else 0)
    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))

    col = 0

    # CT en escala de grises
    axes[col].imshow(ct[:, :, z], cmap="gray", vmin=0, vmax=1)
    axes[col].set_title(f"CT  (z={z})")
    axes[col].axis("off")
    col += 1

    # Máscara predicha
    axes[col].imshow(pred[:, :, z], cmap="gray", vmin=0, vmax=1)
    axes[col].set_title("Predicción")
    axes[col].axis("off")
    col += 1

    # Overlay predicción
    axes[col].imshow(ct[:, :, z], cmap="gray", vmin=0, vmax=1)
    axes[col].imshow(pred[:, :, z], alpha=0.45, cmap="Reds", vmin=0, vmax=1)
    axes[col].set_title("Overlay pred")
    axes[col].axis("off")
    col += 1

    if has_gt:
        # Máscara GT
        axes[col].imshow(gt[:, :, z], cmap="gray", vmin=0, vmax=1)
        axes[col].set_title("GT real")
        axes[col].axis("off")
        col += 1

        # Overlay GT + diferencias: verde=acierto, rojo=FP, azul=FN
        p = (pred[:, :, z] > 0.5).astype(float)
        g = (gt[:, :, z]   > 0.5).astype(float)
        rgb = np.stack([ct[:, :, z]] * 3, axis=-1)
        tp_mask = (p * g)         # True positives  → verde
        fp_mask = (p * (1 - g))   # False positives → rojo
        fn_mask = ((1 - p) * g)   # False negatives → azul
        rgb[:, :, 1] = np.clip(rgb[:, :, 1] + 0.6 * tp_mask, 0, 1)  # G
        rgb[:, :, 0] = np.clip(rgb[:, :, 0] + 0.6 * fp_mask, 0, 1)  # R
        rgb[:, :, 2] = np.clip(rgb[:, :, 2] + 0.6 * fn_mask, 0, 1)  # B
        axes[col].imshow(rgb)
        axes[col].set_title("TP=verde FP=rojo FN=azul")
        axes[col].axis("off")
        col += 1

    if has_prob:
        im = axes[col].imshow(prob[:, :, z], cmap="hot", vmin=0, vmax=1)
        axes[col].set_title("Probabilidades")
        axes[col].axis("off")
        plt.colorbar(im, ax=axes[col], fraction=0.046, pad=0.04)
        col += 1

    plt.tight_layout()
    plt.show()

interact(show_slice, z=widgets.IntSlider(min=0, max=D - 1, step=1, value=D // 2,
                                         description="Corte Z", continuous_update=False,
                                         layout=widgets.Layout(width="60%")));

## Resumen de cortes con predicciones positivas

In [ ]:
# Cortes que contienen al menos un vóxel predicho como nódulo
positive_slices = [z for z in range(D) if pred[:, :, z].sum() > 0]
print(f"Cortes con predicciones positivas ({len(positive_slices)} / {D}): {positive_slices[:30]}")

## Vista en cuadrícula — todos los cortes positivos

In [ ]:
MAX_SHOW = 24  # máximo de cortes a mostrar en la cuadrícula

slices_to_show = positive_slices[:MAX_SHOW]
if not slices_to_show:
    print("No hay cortes con predicciones positivas.")
else:
    cols = 6
    rows = (len(slices_to_show) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    axes = np.array(axes).reshape(-1)

    for i, z in enumerate(slices_to_show):
        axes[i].imshow(ct[:, :, z], cmap="gray", vmin=0, vmax=1)
        axes[i].imshow(pred[:, :, z], alpha=0.4, cmap="Reds", vmin=0, vmax=1)
        axes[i].set_title(f"z={z}", fontsize=8)
        axes[i].axis("off")

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.suptitle("Cortes con predicciones positivas (overlay)", fontsize=12)
    plt.tight_layout()
    plt.show()

## Vista en tres planos (axial / coronal / sagital)

In [ ]:
# Corte central en cada eje
cx, cy, cz = np.array(ct.shape) // 2

# Si hay máscara predicha, centrar en el centroide de la predicción
if pred.sum() > 0:
    coords = np.argwhere(pred > 0.5)
    cx, cy, cz = coords.mean(axis=0).astype(int)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

views = [
    ("Axial (z)",    ct[:, :, cz],   pred[:, :, cz]),
    ("Coronal (y)",  ct[:, cy, :],   pred[:, cy, :]),
    ("Sagital (x)",  ct[cx, :, :],   pred[cx, :, :]),
]

for col_i, (title, ct_slice, pred_slice) in enumerate(views):
    # Fila 0: solo CT
    axes[0, col_i].imshow(ct_slice, cmap="gray", vmin=0, vmax=1)
    axes[0, col_i].set_title(title)
    axes[0, col_i].axis("off")

    # Fila 1: CT + overlay predicción
    axes[1, col_i].imshow(ct_slice, cmap="gray", vmin=0, vmax=1)
    axes[1, col_i].imshow(pred_slice, alpha=0.45, cmap="Reds", vmin=0, vmax=1)
    axes[1, col_i].set_title(f"{title} + pred")
    axes[1, col_i].axis("off")

plt.suptitle("Vista multiplanar centrada en el centroide de la predicción", fontsize=12)
plt.tight_layout()
plt.show()